# TP — Apache Spark : du cours au code

Ce notebook suit le cours **point par point**.
Chaque section du cours a son équivalent pratique ici.

> **Environnement** : Google Colab — exécuter les cellules dans l'ordre.

---
## 0 — Installation

In [ ]:
!pip install pyspark --quiet
print("PySpark installé.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time
import pandas as pd

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField,
                                IntegerType, StringType, DoubleType)
from pyspark.sql.window import Window
from pyspark import StorageLevel
print("Imports OK.")

---
## Section 2 — Architecture & Concepts fondamentaux

### SparkSession et SparkContext

Le **Driver** contient la logique applicative.
`SparkSession` est son point d'entrée depuis Spark 2.0.
`SparkContext` (`sc`) donne accès à l'API RDD.

In [ ]:
spark = (SparkSession.builder
         .appName("TP-Spark")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"Spark {spark.version} démarré")
print(f"Maître      : {sc.master}")
print(f"Cœurs dispo : {sc.defaultParallelism}")
print(f"Spark UI    : http://localhost:4040")

### RDD — Resilient Distributed Dataset

Collection **immuable**, **partitionnée**, **recalculable**.
Deux types d'opérations : **transformations** (lazy) et **actions** (eager).

In [ ]:
# Créer un RDD depuis une liste
rdd = sc.parallelize([10, 25, 3, 47, 8, 99, 14, 66])
print("Type       :", type(rdd))
print("Partitions :", rdd.getNumPartitions())
print("Contenu    :", rdd.collect())   # collect() = action

In [ ]:
# map : transformer chaque élément
rdd_double = rdd.map(lambda x: x * 2)
print("map(x*2)    :", rdd_double.collect())

# filter : garder les éléments qui satisfont la condition
rdd_grands = rdd.filter(lambda x: x > 20)
print("filter(>20) :", rdd_grands.collect())

# flatMap : map + aplatissement
phrases = sc.parallelize(["Spark est rapide", "Spark est distribué"])
mots = phrases.flatMap(lambda p: p.split())
print("flatMap     :", mots.collect())

In [ ]:
# reduceByKey : agréger par clé (réduit AVANT le shuffle)
ventes_rdd = sc.parallelize([
    ("Nord", 300), ("Sud", 150), ("Nord", 200),
    ("Est",  100), ("Sud", 250), ("Est",  180),
])
total_region = (ventes_rdd
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("CA par région :")
for region, ca in total_region.collect():
    print(f"  {region:<8} : {ca}")

### DAG et Lazy Evaluation

Spark construit un **graphe (DAG)** des transformations.
**Aucun calcul** tant qu'il n'y a pas d'action.
Le DAG Scheduler optimise l'ensemble du pipeline avant d'exécuter.

In [ ]:
# Les transformations ne calculent rien
rdd_pipeline = (sc.parallelize(range(1_000_000))
                  .filter(lambda x: x % 2 == 0)    # transformation
                  .map(lambda x: x ** 2))           # transformation

print("Après 2 transformations : rien calculé.")
print("Type :", type(rdd_pipeline))

# L'ACTION déclenche tout le calcul
t0 = time.time()
total = rdd_pipeline.sum()   # action
print(f"
Après sum() : calculé en {time.time()-t0:.3f}s")
print(f"Résultat    : {total:,}")

### Partitionnement

1 partition = 1 tâche exécutée par 1 executor.
Règle : **2 à 4 partitions par cœur CPU**.

In [ ]:
rdd_parts = sc.parallelize(range(100), numSlices=4)
print("Partitions initiales :", rdd_parts.getNumPartitions())

In [ ]:
# Sur un DataFrame (on y reviendra)
df_tmp = spark.range(1000)
print("Avant repartition :", df_tmp.rdd.getNumPartitions())

df_tmp = df_tmp.repartition(8)      # shuffle complet
print("Après repartition :", df_tmp.rdd.getNumPartitions())

df_tmp = df_tmp.coalesce(4)         # réduction sans shuffle
print("Après coalesce    :", df_tmp.rdd.getNumPartitions())

### Cache et Persistance

Mettre un RDD/DataFrame en cache évite de **recalculer le pipeline**
quand il est réutilisé plusieurs fois.

In [ ]:
rdd_calcul = (sc.parallelize(range(500_000))
               .filter(lambda x: x % 3 == 0)
               .map(lambda x: x ** 2))

# Sans cache — recalcule à chaque action
t0 = time.time(); rdd_calcul.count(); t1 = time.time()
print(f"Sans cache — 1er appel  : {t1-t0:.3f}s")

t0 = time.time(); rdd_calcul.count(); t1 = time.time()
print(f"Sans cache — 2ème appel : {t1-t0:.3f}s  (recalcul)")

In [ ]:
rdd_cache = (sc.parallelize(range(500_000))
              .filter(lambda x: x % 3 == 0)
              .map(lambda x: x ** 2)
              .cache())

# 1er appel : calcule + met en cache
t0 = time.time(); rdd_cache.count(); t1 = time.time()
print(f"Avec cache — 1er appel  : {t1-t0:.3f}s  (calcul + cache)")

# 2ème appel : lit depuis le cache
t0 = time.time(); rdd_cache.count(); t1 = time.time()
print(f"Avec cache — 2ème appel : {t1-t0:.3f}s  (lecture cache) ✓")

rdd_cache.unpersist()

---
## Section 3 — Traitement des données

### RDD, DataFrame, Dataset

| | RDD | DataFrame |
|---|---|---|
| Niveau | Bas | Haut |
| Schéma | Aucun | Colonnes typées |
| Optimisation | Manuelle | Catalyst auto |
| Python | ✓ | ✓ |
| Dataset[T] | — | Scala/Java uniquement |

**Règle** : DataFrame par défaut. RDD si la logique est trop complexe.

### Chargement du fichier

In [ ]:
# Placer ventes.csv dans le même dossier que ce notebook
# Sur Colab : Files → Upload ou Drive

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("ventes.csv"))

print(f"Dimensions : {df.count()} lignes × {len(df.columns)} colonnes")
df.printSchema()

In [ ]:
df.show(5)

### `select()` — Sélection de colonnes

In [ ]:
df.select("produit", "qte", "prix").show(4)

In [ ]:
# Renommer à la volée avec alias
df.select(
    F.col("produit"),
    F.col("prix").alias("prix_unitaire")
).show(4)

### `filter()` — Filtrage de lignes

In [ ]:
df.filter(F.col("qte") > 10).show(4)

In [ ]:
# Conditions combinées
df.filter(
    (F.col("qte") > 10) & (F.col("region") == "Nord")
).show(4)

### `withColumn()` — Nouvelle colonne

In [ ]:
df = df.withColumn("montant",
        F.round(F.col("qte") * F.col("prix"), 2))
df.select("produit", "qte", "prix", "montant").show(4)

In [ ]:
df = df.withColumn("volume",
    F.when(F.col("qte") <= 5,  "Faible")
     .when(F.col("qte") <= 15, "Moyen")
     .otherwise("Elevé"))
df.select("produit", "qte", "volume").show(4)

### `groupBy().agg()` — Agrégation

In [ ]:
df.groupBy("categorie").agg(
    F.round(F.sum("montant"),  2).alias("ca_total"),
    F.round(F.avg("prix"),     2).alias("prix_moyen"),
    F.count("*")                 .alias("nb_ventes")
).orderBy("ca_total", ascending=False).show()

In [ ]:
# Pivot : CA par région et trimestre
df.groupBy("region")   .pivot("trimestre", ["T1","T2","T3","T4"])   .agg(F.round(F.sum("montant"), 0))   .orderBy("region")   .show()

### `join()` — Jointure

In [ ]:
objectifs = spark.createDataFrame([
    ("Informatique",  15000.0),
    ("Audio",          5000.0),
    ("Accessoires",    4000.0),
    ("Peripheriques",  3000.0),
], ["categorie", "objectif"])

result = df.join(objectifs, on="categorie")
result.select("produit", "categorie", "montant", "objectif").show(5)

In [ ]:
# Taux d'atteinte de l'objectif par catégorie
ca_cat = df.groupBy("categorie").agg(F.round(F.sum("montant"), 2).alias("ca"))
ca_cat.join(objectifs, on="categorie")       .withColumn("taux_%", F.round(F.col("ca") / F.col("objectif") * 100, 1))       .orderBy(F.col("taux_%").desc())       .show()

### Spark SQL

In [ ]:
df.createOrReplaceTempView("ventes")
objectifs.createOrReplaceTempView("objectifs")
print("Vues créées.")

In [ ]:
spark.sql('''
    SELECT region,
           COUNT(*)              AS nb_ventes,
           ROUND(SUM(montant),2) AS ca_total
    FROM ventes
    WHERE qte > 5
    GROUP BY region
    ORDER BY ca_total DESC
''').show()

In [ ]:
spark.sql('''
    SELECT v.categorie,
           ROUND(SUM(v.montant), 2) AS ca,
           o.objectif,
           ROUND(SUM(v.montant) / o.objectif * 100, 1) AS taux_pct
    FROM ventes v
    JOIN objectifs o ON v.categorie = o.categorie
    GROUP BY v.categorie, o.objectif
    ORDER BY taux_pct DESC
''').show()

### Fonctions de fenêtre — `Window`

In [ ]:
# Rang par région (montant décroissant)
w = Window.partitionBy("region").orderBy(F.col("montant").desc())
df.withColumn("rang", F.rank().over(w))   .select("region", "produit", "montant", "rang")   .filter(F.col("rang") <= 3)   .orderBy("region", "rang")   .show(15)

In [ ]:
# Cumul du CA par vendeur dans le temps
w2 = Window.partitionBy("vendeur").orderBy("date")            .rowsBetween(Window.unboundedPreceding, 0)
df.withColumn("ca_cumul", F.round(F.sum("montant").over(w2), 2))   .select("vendeur", "date", "montant", "ca_cumul")   .orderBy("vendeur", "date")   .show(8)

### Lecture et Écriture

**Schéma explicite** : toujours en production (`inferSchema` lit deux fois).

In [ ]:
# Schéma explicite
schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("produit",   StringType(),  True),
    StructField("categorie", StringType(),  True),
    StructField("qte",       IntegerType(), True),
    StructField("prix",      DoubleType(),  True),
    StructField("region",    StringType(),  True),
    StructField("vendeur",   StringType(),  True),
    StructField("date",      StringType(),  True),
    StructField("trimestre", StringType(),  True),
])

df_schema = spark.read.schema(schema).option("header", True).csv("ventes.csv")
df_schema.printSchema()
df_schema.show(3)

In [ ]:
# Écriture en Parquet (format recommandé)
df.write.mode("overwrite").parquet("output/ventes_parquet/")
print("Écrit en Parquet.")

# Relire
df_parquet = spark.read.parquet("output/ventes_parquet/")
print(f"Relu depuis Parquet : {df_parquet.count()} lignes")
df_parquet.show(3)

### Bonnes pratiques de performance

Filtrer tôt, sélectionner uniquement les colonnes utiles,
utiliser Parquet, activer AQE, broadcast pour les petites tables.

In [ ]:
# Activer AQE (Spark 3+)
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE activé :", spark.conf.get("spark.sql.adaptive.enabled"))

In [ ]:
# Broadcast join : évite le shuffle sur la petite table
from pyspark.sql.functions import broadcast

result_broadcast = df.join(broadcast(objectifs), on="categorie")
result_broadcast.select("produit", "categorie", "montant", "objectif").show(4)

---
## Section 4 — Composants avancés

### MLlib — Pipeline de classification

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Préparer les données pour la classification (vendeur → catégorie)
df_ml = df.select("qte", "prix", "montant", "categorie")

# Encoder la cible
indexer = StringIndexer(inputCol="categorie", outputCol="label")

assembler = VectorAssembler(
    inputCols=["qte", "prix", "montant"],
    outputCol="features"
)
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label", numTrees=50, seed=42
)
pipeline = Pipeline(stages=[indexer, assembler, rf])

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

acc = MulticlassClassificationEvaluator(metricName="accuracy")         .evaluate(model.transform(test))
print(f"Accuracy RandomForest : {acc:.3f}")

### Structured Streaming — Principe

Traiter un **flux continu** avec la même API DataFrame.
Le flux est une table infinie où de nouvelles lignes s'ajoutent continuellement.

```python
# Lecture d'un flux Kafka
stream = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", "host:9092") \
    .option("subscribe", "events").load()

# Agrégation par fenêtre de 5 min
agg = df.groupBy(window("ts", "5 minutes"), "action").count()

agg.writeStream.outputMode("update") \
   .option("checkpointLocation", "/tmp/ckpt/") \
   .format("console").start().awaitTermination()
```
> Streaming non exécutable ici — nécessite un broker Kafka.

---
## Fin du TP

```python
spark.stop()
```

In [ ]:
spark.stop()
print("Session Spark fermée.")